In [ ]:
from pinecone import Pinecone
import pickle
import tqdm
from tqdm import tqdm
from embed_model import embed_text
import pandas as pd

pc = Pinecone(api_key="xxx")

In [51]:
df = pd.read_csv("foody_combined_data_final.csv")

In [52]:
# Danh sách gốc để đối chiếu
full_days = ['Chủ nhật', 'Thứ hai', 'Thứ ba', 'Thứ tư', 'Thứ năm', 'Thứ sáu', 'Thứ bảy']

def get_selling_days(off_str):
    # 1. Nếu cột ngày nghỉ trống, nghĩa là bán tất cả các ngày
    if pd.isna(off_str) or off_str.strip() == "":
        return ", ".join(full_days)
    
    # 2. Chuyển chuỗi thành list tạm thời để xử lý logic
    off_list = [d.strip() for d in off_str.split('||')]
    
    # 3. Lọc ra những ngày KHÔNG nằm trong list nghỉ
    selling_days = [day for day in full_days if day not in off_list]
    
    # 4. Trả về chuỗi ngày bán (ngăn cách bởi dấu phẩy hoặc || tùy bạn)
    return ", ".join(selling_days)

# Tạo cột Ngày bán từ cột Ngày nghỉ
df['Ngày bán'] = df['Ngày nghỉ'].apply(get_selling_days)

In [53]:
bins = [-float('inf'), 28, 58, 106, 235, 893, float('inf')]

# 2. Định nghĩa nhãn tương ứng cho 6 nhóm
view_labels = [
    'rất ít lượt xem', 
    'Ít lượt xem', 
    'Lượt xem trung bình', 
    'Nhiều lượt xem', 
    'Rất nhiều lượt xem', 
    'Thịnh hành'
]

# 3. Sử dụng pd.cut để chia khoảng
df['View Category'] = pd.cut(
    df['TotalView'], 
    bins=bins, 
    labels=view_labels,
    include_lowest=True
)

In [54]:
# Thay thế chuỗi " || " bằng ", "
df['LstTargetAudience'] = df['LstTargetAudience'].str.replace(' || ', ', ', regex=False)
df['LstCategory'] = df['LstCategory'].str.replace(' || ', ', ', regex=False)

In [55]:
print(df.iloc[0])

RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
MetaKeywords                                        Trà sữa, Trà Chanh
Cuisines                                                      Món Việt
LstTargetAudience                         Sinh viên, Cặp đôi, Nhóm hội
LstCategory                                Ăn vặt/vỉa hè, Café/Dessert
RestaurantUrl        ho-chi-minh/con-cu-co-cacao-chuyen-tra-sua-tu-...
Vị trí                                                             0.0
Giá cả                                                             0.0
Chất l

In [56]:
df["View Category"] = df["View Category"].astype(str)
df = df.fillna("")

for col in df.columns:
    df[col] = df[col].astype(str)

df["RestaurantID"] = df["RestaurantID"].apply(lambda x: str(int(float(x))))

In [63]:
def row_to_text(row):
    name = row["Name"]
    district = row["District"]
    area = row["Area"]

    price_min = row["PriceMin"]
    price_max = row["PriceMax"]

    meta_keywords = row["MetaKeywords"]
    cuisines = row["Cuisines"]
    target_audience = row["LstTargetAudience"]
    category = row["LstCategory"]

    view = row["View Category"]

    home_delivery = "Có giao hàng tận nơi" if row["Giao tận nơi"] == "True" else "Không có giao hàng tận nơi"
    oder_table = "Có đặt bàn" if row["Đặt bàn"] == "True" else "Không có đặt bàn"

    day_on = row["Ngày bán"]
    
    return f"""
    Nhà hàng/Quán {name}
    Nằm ở {district} trong khu vực {area}

    Giá từ {price_min} đến {price_max} VND

    Từ khóa: {meta_keywords}
    Ẩm thực: {cuisines}
    Đối tượng: {target_audience}
    Danh mục: {category}

    Dịch vụ:
    - {home_delivery}
    - {oder_table}

    Lượt xem: {view}
    Ngày bán: {day_on}

    """

In [64]:
test = row_to_text(df.iloc[0])
print(test)


    Nhà hàng/Quán Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi - Đoàn Văn Bơ
    Nằm ở Quận 4 trong khu vực 

    Giá từ 0.0 đến 0.0 VND

    Từ khóa: Trà sữa, Trà Chanh
    Ẩm thực: Món Việt
    Đối tượng: Sinh viên, Cặp đôi, Nhóm hội
    Danh mục: Ăn vặt/vỉa hè, Café/Dessert

    Dịch vụ:
    - Không có giao hàng tận nơi
    - Không có đặt bàn

    Lượt xem: rất ít lượt xem
    Ngày bán: Chủ nhật, Thứ hai, Thứ ba, Thứ tư, Thứ năm, Thứ sáu, Thứ bảy

    


In [66]:
# Nhân
# start = 0
# end = 30000
# texts = [row_to_text(row) for _, row in df.iloc[start:end].iterrows()]

# start = 9632
# end = 30000
# texts = [row_to_text(row) for _, row in df.iloc[start:end].iterrows()]
 
# Huy
start = 0
end = 85
texts = [row_to_text(row) for _, row in df.iloc[start:end].iterrows()]

In [67]:
index = pc.Index("restaurant")

In [68]:
def clean_text(text):
    return text.encode("utf-8", "ignore").decode("utf-8")

In [69]:
batch_size = 16

for i in tqdm(range(0, len(texts), batch_size)):
    batch_names = []
    batch_ids = []
    batch_texts = []

    for j in range(i, min(i + batch_size, len(texts))):
        row = df.iloc[j + start]

        text = row_to_text(row)
        batch_texts.append(text)

        batch_names.append(row["Name"])
        batch_ids.append(row["RestaurantID"])
    
    if len(batch_names) == 0:
        continue 

    batch_emb = embed_text(batch_texts).cpu().numpy()

    vectors = [
        {
            "id": batch_ids[k],
            "values": batch_emb[k].tolist(),
            "metadata": {
                "text": clean_text("Nhà hàng: " + batch_names[k]),
            }
        }
        for k in range(len(batch_ids))
    ]

    index.upsert(vectors=vectors)

    del batch_emb, vectors

100%|██████████| 6/6 [00:52<00:00,  8.69s/it]
